In [19]:
from torch import tensor, nn
import torch

### Adam vs SGD

#### SGD

In [141]:
model = nn.Linear(1,1, bias = False)
model.load_state_dict({
    "weight": torch.tensor([[1]]),
})

<All keys matched successfully>

In [142]:
model.weight

Parameter containing:
tensor([[1.]], requires_grad=True)

In [143]:
loss_fn = nn.MSELoss()

In [144]:
x = tensor([10.0])
y_true = tensor([15.0])

In [145]:
y_pred = model(x)

In [146]:
y_pred

tensor([10.], grad_fn=<SqueezeBackward4>)

In [147]:
loss = loss_fn(y_pred, y_true)
loss

tensor(25., grad_fn=<MseLossBackward0>)

In [148]:
(15.0 - 10.0) ** 2

25.0

In [149]:
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.001
)

In [150]:
optimizer.state

defaultdict(dict, {})

In [153]:
loss.backward()

In [154]:
model.weight.grad

tensor([[-100.]])

In [155]:
model.weight

Parameter containing:
tensor([[1.]], requires_grad=True)

In [156]:
optimizer.step()

In [157]:
optimizer.state

defaultdict(dict, {})

In [158]:
model.weight

Parameter containing:
tensor([[1.1000]], requires_grad=True)

In [159]:
1 - 0.001 * (-100)

1.1

In [160]:
optimizer.zero_grad()

In [161]:
y_pred = model(x)
loss = loss_fn(y_pred, y_true)
print(loss)
loss.backward()
print(model.weight.grad)
optimizer.step()
optimizer.zero_grad()

tensor(16., grad_fn=<MseLossBackward0>)
tensor([[-80.]])


In [162]:
model.weight

Parameter containing:
tensor([[1.1800]], requires_grad=True)

In [163]:
1.1 - 0.001 * (-80)

1.1800000000000002

In [164]:
optimizer.state

defaultdict(dict, {})

#### Adam

In [278]:
model = nn.Linear(1,1, bias = False)
model.load_state_dict({
    "weight": torch.tensor([[1]]),
})
loss_fn = nn.MSELoss()
x = tensor([10.0])
y_true = tensor([15.0])

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [279]:
beta1, beta2 = optimizer.defaults["betas"]
beta1, beta2 

(0.9, 0.999)

In [280]:
optimizer.state

defaultdict(dict, {})

In [281]:
y_pred = model(x)
loss = loss_fn(y_pred, y_true)
print("Loss:", loss)
loss.backward()
print("Model Grad:", model.weight.grad)
optimizer.step()
optimizer.zero_grad()
print("Model Weight:", model.weight)

Loss: tensor(25., grad_fn=<MseLossBackward0>)
Model Grad: tensor([[-100.]])
Model Weight: Parameter containing:
tensor([[1.0010]], requires_grad=True)


In [282]:
def calculate_(prev_m, prev_v, old_weight, grad, t):
    current_m = beta1 * prev_m + (1 - beta1) * grad
    current_v = beta2 * prev_v + (1 - beta2) * grad ** 2
    
    current_m = (current_m) / (1 - beta1 ** t)
    current_v = (current_v) / (1 - beta2 ** t)
    eps = 1e-8
    
    return current_m / ((current_v) ** (1/2) + eps)

In [286]:
calculate_(0, 0, 1., -100, 1)

-0.9999999999000001

In [288]:
1 - 0.001 * calculate_(0, 0, 1., -100, 1)

1.0009999999999

In [289]:
optimizer.state

defaultdict(dict,
            {Parameter containing:
             tensor([[1.0010]], requires_grad=True): {'step': tensor(1.),
              'exp_avg': tensor([[-10.]]),
              'exp_avg_sq': tensor([[10.]])}})

In [290]:
param = next(model.parameters())
param

Parameter containing:
tensor([[1.0010]], requires_grad=True)

In [291]:
m = optimizer.state[param]['exp_avg']
v = optimizer.state[param]['exp_avg_sq']
step = optimizer.state[param]['step']

In [292]:
step

tensor(1.)

In [293]:
m.item(), v.item()

(-10.0, 10.0)

In [294]:
optimizer.state.keys()

dict_keys([Parameter containing:
tensor([[1.0010]], requires_grad=True)])

In [295]:
y_pred = model(x)
loss = loss_fn(y_pred, y_true)
print("Loss:", loss)
loss.backward()
print("Model Grad:", model.weight.grad)

Loss: tensor(24.9001, grad_fn=<MseLossBackward0>)
Model Grad: tensor([[-99.8000]])


In [296]:
model.weight.item() - 0.001 * calculate_(m.item(), v.item(), model.weight.item(), model.weight.grad.item(), 2)

1.001999994044209

In [297]:
optimizer.step()
print("Model Weight:", model.weight)

Model Weight: Parameter containing:
tensor([[1.0020]], requires_grad=True)


In [298]:
optimizer.zero_grad()

In [299]:
model.weight.item()

1.0019999742507935

In [300]:
optimizer.state

defaultdict(dict,
            {Parameter containing:
             tensor([[1.0020]], requires_grad=True): {'step': tensor(2.),
              'exp_avg': tensor([[-18.9800]]),
              'exp_avg_sq': tensor([[19.9500]])}})